# 코드 설명

- **코드 파일 사용 목차** : 주성분분석(PCA)으로 데이터의 차원 줄이기
- **해당 코드에서 사용하는 CSV 파일** : `file_pe_headers.csv`
    - 규모 : 19,611행 × 79열
    - 내용 : Windows 실행 파일(.exe, .dll)의 PE 헤더 필드를 추출한 데이터
    - 목표 변수 : `Malware` (1이면 악성, 0이면 정상). 본 파일에서는 사용하지 않음
    - 식별자 열 : `Name` (파일명)
    - 설명 변수 : 나머지 77열. 전부 수치형
    - 결측 : 없음
- **사용 라이브러리**
    - `pandas` (as pd) — CSV 적재 및 표 형식 데이터 처리
    - `sklearn` — 전처리 및 차원 축소
- **사용되는 함수**
    - `pd.read_csv("파일경로", sep=",")`
        - 함수의 목적 : CSV 파일을 `DataFrame` 객체로 적재
        - 함수의 파라미터
            - `filepath_or_buffer` : 파일 경로. 첫 번째 위치 인자
            - `sep` : 열 구분자. 기본값이 `,`이므로 본 코드에서는 생략 가능
    - `DataFrame.drop(labels, axis)`
        - 함수의 목적 : 지정한 행 또는 열을 제거한 새 `DataFrame` 반환
        - 함수의 파라미터
            - `labels` : 제거 대상 이름. 여러 개는 리스트로 전달
            - `axis` : `0`이면 행, `1`이면 열. 기본값 `0`
    - `DataFrame.to_numpy()`
        - 함수의 목적 : `DataFrame`을 NumPy 배열(`ndarray`)로 변환
    - `StandardScaler()`
        - 함수의 목적 : 각 열을 평균 0, 표준편차 1로 변환. PCA 수행 전 필수 단계
        - 주요 메서드 : `fit(X)`, `transform(X)`, `fit_transform(X)`
    - `PCA(n_components)`
        - 함수의 목적 : 데이터의 분산을 가장 잘 설명하는 새 축을 순서대로 찾는 객체 생성
        - 함수의 파라미터
            - `n_components` : 남길 주성분 개수. 미지정 시 가능한 최대 개수
                - 정수 지정 시 해당 개수, `0.90`처럼 소수 지정 시 누적 설명 비율 기준 자동 결정
            - `random_state` : 근사 알고리즘 사용 시의 난수 시드
        - 주요 메서드 : `fit(X)`, `fit_transform(X)`
        - 주요 속성
            - `explained_variance_ratio_` : 각 주성분이 전체 분산에서 차지하는 비율
            - `components_` : 각 주성분을 구성하는 원래 변수들의 가중치
- **실행 순서** : 적재 → 설명 변수 X 구성 → 표준화 → PCA 수행 → 설명 비율 확인

In [1]:
# ── Google Colab 준비 셀 ─────────────────────────────────────────
# 로컬(VS Code, JupyterLab)에서 실행하면 아무 동작 없이 넘어갑니다.
# Colab에서 실행하면 저장소를 내려받아 실습 폴더로 이동합니다.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    저장소 = "https://github.com/isl-hedgehog/python-cyber-seminar.git"
    이름 = 저장소.rsplit("/", 1)[-1].removesuffix(".git")
    실습폴더 = "01-data-preprocessing"

    if not os.path.isdir(f"/content/{이름}"):
        subprocess.run(["git", "clone", "-q", 저장소], cwd="/content", check=True)
    os.chdir(f"/content/{이름}/{실습폴더}")

    # Colab에는 pandas, numpy, scikit-learn이 이미 설치되어 있습니다.
    # 추가 패키지가 필요한 회차에서는 아래 줄의 주석을 해제하세요.
    # subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)

    print("Colab 준비 완료 :", os.getcwd())
else:
    print("로컬 환경 — 준비 셀을 건너뜁니다.")

로컬 환경 — 준비 셀을 건너뜁니다.


In [2]:
from sklearn.decomposition import PCA
import pandas as pd

data = pd.read_csv("file_pe_headers.csv", sep=",")
X = data.drop(["Name", "Malware"], axis=1).to_numpy()

### 1. PE 헤더 데이터 적재 및 X 구성

**데이터의 성격**

`file_pe_headers.csv`는 Windows 실행 파일의 **PE(Portable Executable) 헤더** 값을 추출해 표로 정리한 데이터. 파일을 실행하지 않고 구조 정보만으로 악성 여부를 판별하는 **정적 분석** 방식에 사용.

표본 19,611개, 설명 변수 77개.

**두 열을 제외하는 이유**

| 열 | 제외 사유 |
|---|---|
| `Malware` | 정답 열. X에 포함되면 모델이 답을 그대로 보게 됨 |
| `Name` | 파일명. 각 행을 구분하는 이름표일 뿐 예측에 쓸 정보가 아님 |

파일명은 행마다 값이 전부 다르므로 훈련 데이터를 통째로 외우는 것이 가능하며, 이는 규칙을 학습한 결과가 아님. 또한 악성과 정상 검체를 서로 다른 경로에서 수집한 경우 명명 규칙 자체가 정답과 연관되어 **데이터 누수**로 이어질 가능성 존재.

**본 파일의 위치**

본 파일은 정답을 맞히는 지도학습이 아니라, 정답 없이 데이터 구조만 파악하는 **비지도 차원 축소**를 목적으로 함. 따라서 `Malware` 열은 이후 단계에서 전혀 사용하지 않으며, X의 77개 변수가 서로 어떻게 얽혀 있는지만 분석 대상.

In [3]:
from sklearn.preprocessing import StandardScaler

X_standardized = StandardScaler().fit_transform(X)

### 2. PCA 수행 전 표준화

**하는 일**

각 열의 평균 $\mu$와 표준편차 $\sigma$를 구한 뒤 다음 식으로 값을 변환.

$$z = \frac{x - \mu}{\sigma}$$

변환 후 모든 열이 평균 0, 표준편차 1을 가짐.

**PCA에서 표준화가 특히 중요한 이유**

다른 모델에서 표준화는 학습 속도 개선 정도의 이점에 그치지만, **PCA에서는 결과 자체를 바꾸는 요인**.

PCA는 분산이 큰 방향을 순서대로 찾는 방법. 그런데 값의 범위가 큰 열은 그 자체로 분산이 크게 나옴. PE 헤더에서 `SizeOfImage`는 백만 단위 바이트 값이고 `NumberOfSections`는 보통 한 자릿수이므로, 표준화를 건너뛰면 첫 번째 주성분이 사실상 `SizeOfImage` 방향으로 결정. 이는 변수들 사이의 관계가 아니라 **단위를 무엇으로 쟀는지**를 반영한 결과에 불과.

| 표준화 | 결과의 성질 |
|---|---|
| 미실시 | 열의 측정 단위가 바뀌면 주성분도 함께 바뀜 |
| 실시 | 단위와 무관하게 동일한 결과 |

변수의 단위가 서로 다른 데이터에서는 표준화 후 PCA가 표준적인 선택이며, 본 데이터가 이에 해당.

**보완 사항**

현재 코드는 데이터를 나누기 전 전체 X에 `fit_transform`을 적용. 축소 결과를 이후 분류 모델의 입력으로 쓸 경우, 평균·표준편차와 주성분 방향에 테스트 데이터의 정보가 섞여 **전처리 누수** 발생. 본 파일은 데이터 구조 파악만이 목적이므로 문제되지 않으나, 예측 모델을 만들 때는 `Pipeline`으로 묶어 훈련 부분에만 `fit`이 적용되도록 구성 필요.

In [4]:
pca = PCA()
pca.fit_transform(X_standardized)

array([[-1.16886290e-01, -7.15480741e-02,  3.16916194e+00, ...,
        -7.32591189e-15, -2.21553954e-15,  4.22450907e-15],
       [-3.04037254e-01, -1.28517084e-01, -2.40204566e-01, ...,
        -4.40869250e-15, -1.42713092e-15,  2.24630452e-15],
       [-3.07312786e-01, -1.31296697e-01, -1.06253036e+00, ...,
        -9.09501369e-15, -2.75468852e-15,  5.11763444e-15],
       ...,
       [ 6.24269798e-01, -5.07561736e-02,  6.13792253e-01, ...,
        -2.24516335e-15, -7.07133968e-16,  5.87787337e-16],
       [-2.54202111e-01, -2.25874904e-02,  1.53710266e-01, ...,
         2.61238357e-15,  7.14988880e-16, -1.71850955e-15],
       [-4.06052243e-01, -4.98937892e-02, -3.67792815e-02, ...,
         7.14735207e-15,  2.22034239e-15, -4.15776270e-15]],
      shape=(19611, 77))

### 3. 주성분분석 수행

**주성분이란**

PCA는 기존 77개 변수를 조합해 **새로운 축**을 만드는 방법. 축을 만드는 규칙은 두 가지.

1. 첫 번째 축은 데이터가 가장 넓게 퍼진 방향으로 설정
2. 그다음 축은 앞선 축과 직각을 이루면서 남은 분산이 가장 큰 방향으로 설정

이렇게 만들어진 각 축이 **주성분**이며, 원래 변수들에 각각 가중치를 곱해 더한 값. 앞쪽 주성분일수록 더 많은 정보를 담고 있으므로, 뒤쪽 몇 개를 버려도 손실이 적다는 것이 차원 축소의 원리.

**`PCA()`에 인자를 주지 않은 경우**

`n_components`가 미지정이므로 만들 수 있는 주성분을 **전부**, 즉 77개 생성. 이 시점에서는 차원이 줄지 않으며 77차원을 77차원으로 축만 바꾼 상태.

이는 곧바로 축소하지 않고 **분산이 어떻게 분포하는지 먼저 확인**하려는 통상적 절차. 전체 목록을 확보한 뒤 다음 셀에서 몇 개를 남길지 결정.

**출력 배열 끝의 극히 작은 값**

만들어진 77개 중 **8개는 설명하는 분산이 0**. 원본 데이터에 모든 행의 값이 같은 열이 8개 있어 실질적인 정보가 69개분밖에 없기 때문. 이 8개 주성분의 값은 계산 과정에서 생긴 $10^{-15}$ 수준의 오차이며 의미 없음. 출력 배열 오른쪽 끝에 보이는 `-7.32e-15` 같은 값이 이에 해당.

**주의 — 반환값 미할당**

`fit_transform`의 결과를 변수에 담지 않아 **변환된 좌표가 버려짐**. 본 셀은 `pca` 객체를 학습시키는 효과만 활용한 상태이므로, 목적이 학습뿐이라면 `pca.fit(X_standardized)`가 의도를 명확히 표현. 변환 결과가 이후 필요하면 `X_pca = pca.fit_transform(X_standardized)`로 할당 필요.

In [5]:
print(pca.explained_variance_ratio_)
print(f"{sum(pca.explained_variance_ratio_[ : 40])*100:6.4f}%")

[1.13714096e-01 6.04526312e-02 5.35847638e-02 4.95286930e-02
 4.08242868e-02 3.43687925e-02 3.32004002e-02 3.01112226e-02
 2.86901095e-02 2.81624164e-02 2.54807940e-02 2.38845548e-02
 2.22696648e-02 2.05755591e-02 1.82485433e-02 1.73648310e-02
 1.66649078e-02 1.63647194e-02 1.52683994e-02 1.46357930e-02
 1.45790542e-02 1.45535760e-02 1.44699413e-02 1.44154480e-02
 1.42948516e-02 1.39221004e-02 1.35338124e-02 1.33766277e-02
 1.32896667e-02 1.23472302e-02 1.20507834e-02 1.15452214e-02
 1.13731313e-02 1.10939084e-02 1.07062189e-02 1.01649154e-02
 9.90148375e-03 9.61478385e-03 9.17627698e-03 9.04802544e-03
 8.66332999e-03 6.94752252e-03 6.84216033e-03 6.48244001e-03
 5.95005317e-03 5.91335216e-03 5.41615029e-03 5.10640740e-03
 4.83543074e-03 4.45888820e-03 4.29104432e-03 3.82076025e-03
 3.79864324e-03 3.24146447e-03 3.18558571e-03 2.67004617e-03
 2.03201471e-03 1.73591476e-03 1.65758475e-03 1.56708821e-03
 1.38839592e-03 1.20694096e-03 8.20896559e-04 6.92520065e-04
 2.79632267e-04 1.366147

### 4. 설명 분산 비율 확인 및 주성분 개수 결정

**`explained_variance_ratio_`의 의미**

각 주성분이 전체 정보량 중 몇 퍼센트를 담고 있는지를 나타내는 배열. 전부 더하면 1이 되며, 큰 값부터 순서대로 정렬되어 있음. 따라서 앞에서부터 누적하면 해당 개수만큼 남겼을 때 보존되는 정보량 산출.

**본 데이터의 결과**

| 주성분 개수 | 누적 설명 비율 |
|---|---|
| 1 | 11.37% |
| 10 | 47.26% |
| 20 | 66.34% |
| **40** | **90.69%** |
| 47 | 95.00% |
| 69 | 100.00% |

첫 번째 주성분이 11.37%에 그치고 이후 완만하게 감소. 특정 소수의 변수가 데이터를 좌우하지 않고 **정보가 여러 방향에 고루 퍼져 있다는 의미**이며, 과감한 차원 축소가 어려운 형태.

**40이라는 숫자의 근거**

코드의 `[:40]`이 만들어 내는 90.69%는 임의로 정한 값이 아님. 누적 비율이 90%를 넘는 **최소 개수가 정확히 40개**(39개는 89.78%). 차원을 77개에서 40개로 약 절반 줄이면서 정보의 9할을 보존한다는 뜻이며, 90 · 95 · 99%는 관행적으로 쓰이는 기준선.

이 계산은 다음과 같이 자동화 가능.

```python
import numpy as np
누적 = np.cumsum(pca.explained_variance_ratio_)
k = int(np.searchsorted(누적, 0.90) + 1)   # 40
```

또는 `PCA(n_components=0.90)`으로 비율을 직접 지정하면 동일한 결과.

**개수를 정하는 다른 기준**

| 기준 | 내용 | 본 데이터 적용 시 |
|---|---|---|
| 누적 비율 | 90 / 95 / 99% 도달 지점 | 40 / 47 / 57 |
| 카이저 규칙 | 설명 분산이 평균 이상인 주성분만 | 22 |
| 스크리 도표 | 그래프 기울기가 꺾이는 지점 | 육안 판단 |

기준마다 결과 차이가 크므로, 최종 개수는 수치 기준만이 아니라 이후 작업에 필요한 성능과 계산 비용을 함께 고려해 결정.

**참고 — 해석의 한계**

주성분은 여러 변수를 섞어 만든 축이므로 "이 축이 무엇을 의미하는가"를 설명하기 어려움. 차원이 줄어 계산은 빨라지지만 **변수 단위의 해석은 불가능**. 어떤 헤더 필드가 악성 판별에 기여하는지가 관심사라면, PCA보다 원래 변수를 그대로 남기는 특징 선택 방식이 적합.

**출력 서식**

`{...:6.4f}`는 최소 6칸 확보, 소수점 아래 4자리 표기 지정. 다만 값이 `90.6852`로 이미 7자리이므로 칸 수 지정의 효과는 없음.